# Лекция: Статистические критерии в Python

**Дисциплина:** Введение в анализ больших данных

Темы:
- Shapiro–Wilk (нормальность);
- критерий Фишера (равенство дисперсий);
- Kolmogorov–Smirnov (одна и две выборки);
- t-тест Стьюдента;
- критерий Пирсона (хи-квадрат) для таблиц сопряжённости;
- мозаичные диаграммы / heatmap частот.

Инструменты: **scipy.stats**, **pandas**, **matplotlib**, **seaborn**, **statsmodels**.

Демо-данные: **tips**, **penguins** и синтетические выборки. Примеры **не совпадают** с лабораторным заданием — его выполните самостоятельно.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.mosaicplot import mosaic

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Shapiro–Wilk: нормальность

H0: выборка из нормального закона.  
При p > 0.05 обычно **не отвергают** H0.  
Объём: примерно от 3 до 5000.


In [ ]:
x_norm = stats.norm.rvs(loc=10, scale=2, size=80)
x_unif = stats.uniform.rvs(loc=0, scale=20, size=80)

for name, x in [("нормальная", x_norm), ("равномерная", x_unif)]:
    W, p = stats.shapiro(x)
    verdict = "не отвергаем H0" if p > 0.05 else "отвергаем H0"
    print(f"{name:12s}: W={W:.4f}, p={p:.4g}  →  {verdict}")


In [ ]:
tips = sns.load_dataset("tips")
print("=== Shapiro–Wilk (tips) ===")
for col in ["total_bill", "tip", "size"]:
    W, p = stats.shapiro(tips[col])
    verdict = "похоже на нормальное" if p > 0.05 else "не нормальное"
    print(f"{col:12s}: W={W:.4f}, p={p:.4g}  →  {verdict}")


---
## 2. Критерий Фишера: равенство дисперсий

H0: $\sigma_1^2 = \sigma_2^2$.  
Статистика $F = s_1^2 / s_2^2$, p-value — двусторонний по F-распределению.


In [ ]:
def f_test(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    F = a.var(ddof=1) / b.var(ddof=1)
    dfn, dfd = len(a) - 1, len(b) - 1
    p = 2 * min(stats.f.cdf(F, dfn, dfd), 1 - stats.f.cdf(F, dfn, dfd))
    return F, p

a = tips.loc[tips["sex"] == "Male", "total_bill"]
b = tips.loc[tips["sex"] == "Female", "total_bill"]
F, p = f_test(a, b)
print(f"total_bill: Male vs Female")
print(f"  F={F:.4f}, p={p:.4g}")
print("  → дисперсии равны" if p > 0.05 else "  → дисперсии различаются")


---
## 3. Kolmogorov–Smirnov

### Две выборки

H0: обе из одного непрерывного распределения.  
`stats.ks_2samp(x, y)`


In [ ]:
u1 = stats.uniform.rvs(0, 1, size=200)
u2 = stats.uniform.rvs(0, 1, size=200)
n1 = stats.norm.rvs(0, 1, size=200)

D, p = stats.ks_2samp(u1, u2)
print(f"U vs U: D={D:.4f}, p={p:.4g}  →  {'одно распределение' if p > 0.05 else 'разные'}")

D, p = stats.ks_2samp(u1, n1)
print(f"U vs N: D={D:.4f}, p={p:.4g}  →  {'одно распределение' if p > 0.05 else 'разные'}")


### Одна выборка: согласие с законом

`stats.kstest(x, "norm", args=(mu, sigma))`  
`stats.kstest(x, "uniform", args=(loc, scale))`


In [ ]:
mu, sigma = x_norm.mean(), x_norm.std(ddof=1)
D, p = stats.kstest(x_norm, "norm", args=(mu, sigma))
print(f"x_norm ~ N(оценка): D={D:.4f}, p={p:.4g}")

D, p = stats.kstest(x_unif, "uniform", args=(0, 20))
print(f"x_unif ~ U(0,20):   D={D:.4f}, p={p:.4g}")


---
## 4. t-тест Стьюдента: равенство средних

H0: $\mu_1 = \mu_2$.  
`stats.ttest_ind(a, b, equal_var=False)` — версия Уэлча (не требует равенства дисперсий).


In [ ]:
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"Средние total_bill: Male vs Female")
print(f"  mean Male={a.mean():.2f}, Female={b.mean():.2f}")
print(f"  t={t:.4f}, p={p:.4g}")
print("  → средние равны" if p > 0.05 else "  → средние различаются")


---
## 5. Хи-квадрат Пирсона: независимость признаков

H0: признаки независимы.  
`stats.chi2_contingency(table)`


In [ ]:
ct = pd.crosstab(tips["day"], tips["time"])
print(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"\nchi2={chi2:.3f}, df={dof}, p={p:.4g}")
print("→ отвергаем H0 (зависимость)" if p < 0.05 else "→ не отвергаем H0 (независимость)")


In [ ]:
ct2 = pd.crosstab(tips["sex"], tips["smoker"])
print(ct2)
chi2, p, dof, _ = stats.chi2_contingency(ct2)
print(f"chi2={chi2:.3f}, p={p:.4g}")


### Мозаика и heatmap частот


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

mos_data = {(str(i), str(j)): ct.loc[i, j] for i in ct.index for j in ct.columns}
mosaic(mos_data, ax=axes[0], title="day × time")

sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("Частоты day × time")
plt.tight_layout()
plt.show()


---
## 6. Краткая сводка по penguins (дополнительно)

Несколько тестов подряд на одном датасете.


In [ ]:
peng = sns.load_dataset("penguins").dropna()
print("Shapiro body_mass_g:")
W, p = stats.shapiro(peng["body_mass_g"])
print(f"  W={W:.4f}, p={p:.4g}")

adel = peng.loc[peng["species"] == "Adelie", "body_mass_g"]
gent = peng.loc[peng["species"] == "Gentoo", "body_mass_g"]
t, p = stats.ttest_ind(adel, gent, equal_var=False)
print(f"\nt-тест массы Adelie vs Gentoo: t={t:.3f}, p={p:.4g}")

ct_p = pd.crosstab(peng["species"], peng["island"])
chi2, p, dof, _ = stats.chi2_contingency(ct_p)
print(f"\nХи-квадрат species × island: chi2={chi2:.2f}, p={p:.4g}")
print(ct_p)


---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Shapiro–Wilk | `stats.shapiro(x)` → (W, p) |
| F-тест дисперсий | `F = s1/s2`; p через `stats.f.cdf` |
| KS две выборки | `stats.ks_2samp(x, y)` |
| KS vs нормальное | `stats.kstest(x, "norm", args=(mu, sigma))` |
| KS vs равномерное | `stats.kstest(x, "uniform", args=(a, b-a))` |
| t-тест средних | `stats.ttest_ind(a, b, equal_var=False)` |
| Хи-квадрат | `stats.chi2_contingency(table)` |
| Таблица частот | `pd.crosstab(A, B)` |
| Мозаика | `statsmodels.graphics.mosaicplot.mosaic` |

---
## Что сделать после лекции

1. Повторите каждый тест на **других** столбцах / своих выборках.  
2. Откройте лабораторное задание и выполните его **самостоятельно** на указанных таблицах.  
3. При интерпретации: p > 0.05 → H0 обычно **не отвергают** (на уровне 0.05).

Удачи!
